## Aggregate LOOCV Results
This script searches through all subfolders of a given input folder (except those named "Archive"), 
finds all files named `loocv_results.csv`, extracts the first row of each file keeping only the columns:
- `stats.bias`
- `stats.MSPE`
- `stats.RMSPE`
- `stats.RAV`


In [99]:

import os
import pandas as pd

# Set the path to your input folder (adjust as needed)

input_folders = ['CPF', 'ETF'] 
output_file = 'Results Summary\loocv_summary.csv'

# List to collect summary rows
summary_rows = []

for input_folder in input_folders:
    # Walk through the input folder recursively
    for root, dirs, files in os.walk(input_folder):
        # Skip any subfolder named "Archive" (case insensitive)
        dirs[:] = [d for d in dirs if d.lower() != "archive"]
        
        if "loocv_results.csv" in files:
            print(f"Processing {root}")
            file_path = os.path.join(root, "loocv_results.csv")
            
            # Compute the relative path from the input folder
            rel_path = os.path.normpath(os.path.relpath(file_path, input_folder))
            rel_parts = rel_path.split(os.sep)
            
            # Ensure there are at least three folder levels: [..., segment_interval, independent_variable, file]
            if len(rel_parts) < 3:
                continue
            
            # Extract the folder names:
            # - Two folders up (index -3) for Segment Interval
            # - One folder up (index -2) for independent variable
            segment_interval_folder = rel_parts[-3]
            independent_variable_folder = rel_parts[-2]
            
            # Process folder names by splitting on whitespace and taking all tokens except the first one.
            seg_tokens = segment_interval_folder.split()
            segment_interval = " ".join(seg_tokens[1:]) if len(seg_tokens) > 1 else segment_interval_folder
            
            ind_tokens = independent_variable_folder.split()
            independent_variable = " ".join(ind_tokens[1:]) if len(ind_tokens) > 1 else independent_variable_folder
            watershed = ind_tokens[0] if len(ind_tokens) > 0 else "Unknown"
            # The Region is taken as the base name of the input folder.
            region = os.path.basename(os.path.normpath(input_folder))
            
            try:
                # Read the CSV and extract the first row
                df = pd.read_csv(file_path)
                if df.empty:
                    continue
                first_row = df.iloc[0]
                
                # Select the desired columns
                selected_columns = ["stats.bias", "stats.MSPE", "stats.RMSPE", "stats.RAV"]
                # If any column is missing, an error will be raised.
                selected_data = first_row[selected_columns]
                
                # Build a dictionary for this row
                row_data = {
                    "Region": region,
                    "Watershed": watershed,
                    "Segment Interval": segment_interval,
                    "Independent Variable": independent_variable,
                    "Bias": selected_data["stats.bias"],
                    "MSPE": selected_data["stats.MSPE"],
                    "RMSPE": selected_data["stats.RMSPE"],
                    "RAV": selected_data["stats.RAV"],
                }
                summary_rows.append(row_data)
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

# Create a DataFrame from the collected rows and write to CSV
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(output_file, index=False)
    print(f"Summary saved to {output_file}")
else:
    print("No valid files found.")


Processing CPF\Outputs\Segmented 10m\CPF sfm_deposition_logtrans
Processing CPF\Outputs\Segmented 10m\CPF sfm_erosion_logtrans
Processing CPF\Outputs\Segmented 10m\CPF sfm_net change_logtrans
Processing CPF\Outputs\Segmented 10m\ME sfm_deposition_logtrans
Processing CPF\Outputs\Segmented 10m\ME sfm_erosion_logtrans
Processing CPF\Outputs\Segmented 10m\ME sfm_net change_logtrans
Processing CPF\Outputs\Segmented 10m\MM lidar_erosion_logtrans
Processing CPF\Outputs\Segmented 10m\MM sfm_deposition_logtrans
Processing CPF\Outputs\Segmented 10m\MM sfm_erosion_logtrans
Processing CPF\Outputs\Segmented 10m\MM sfm_net change_logtrans
Processing CPF\Outputs\Segmented 10m\MW sfm_deposition_logtrans
Processing CPF\Outputs\Segmented 10m\MW sfm_erosion_logtrans
Processing CPF\Outputs\Segmented 10m\MW sfm_net change_logtrans
Processing CPF\Outputs\Segmented 10m\UE lidar_erosion_logtrans
Processing CPF\Outputs\Segmented 10m\UE sfm_deposition_logtrans
Processing CPF\Outputs\Segmented 10m\UE sfm_erosion

## Aggregate glance, influence, tidy, and varcomp results to a single csv file

In [100]:
import os
import pandas as pd

# List of input folders to search.
input_folders = ["CPF", "ETF"]  

# Mapping of target result file names to their summary file names.
target_files = {
    "glance_results.csv": r"Results Summary\glance_summary.csv",
    "influence_results.csv": r"Results Summary\influence_summary.csv",
    "tidy_results.csv": r"Results Summary\tidy_summary.csv",
    "varcomp_results.csv": r"Results Summary\varcomp_summary.csv",
    "loocv_results.csv": r"Results Summary\loocv_prediction_summary.csv"
}


# Initialize a dictionary to hold lists of DataFrames for each result type.
summary_data = { key: [] for key in target_files.keys() }

for base_folder in input_folders:
    # Walk through the directory tree.
    for root, dirs, files in os.walk(base_folder):
        # Skip subdirectories named "Archive"
        if "Archive" in dirs:
            dirs.remove("Archive")
            
        # Process each file in the current directory.
        for file in files:
            if file in target_files:
                print(f"Processing root: {root}, file: {file}")
                filepath = os.path.join(root, file)
                #print(f"Processing file: {filepath}")
                
                # Try to read the CSV file.
                try:
                    df = pd.read_csv(filepath)
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    continue


                # Compute metadata using the relative path from the base folder.
                relative_path = os.path.relpath(filepath, start=base_folder)
                parts = relative_path.split(os.path.sep)
                
                # 'Region' is defined as the name of the input folder (base folder).
                region = os.path.basename(os.path.normpath(base_folder))
                
                # For the following, we expect the file to be at least three levels deep.
                # In the example: "CPF/Outputs/Segmented 5m/CPF lidar erosion/glance_results.csv":
                #   - parts[-3] is "Segmented 5m" (Segment Interval),
                #   - parts[-2] is "CPF lidar erosion" (independent variable).
                if len(parts) >= 3:
                    seg_interval_full = parts[-3]
                    indep_var_full = parts[-2]
                else:
                    seg_interval_full = ""
                    indep_var_full = ""
                
                # For "Segment Interval": split the folder name by whitespace and take from the second token onward.
                seg_interval_tokens = seg_interval_full.split()
                segment_interval = " ".join(seg_interval_tokens[1:]) if len(seg_interval_tokens) > 1 else seg_interval_full
                
                # For "independent variable": do the same.
                indep_var_tokens = indep_var_full.split()
                independent_variable = " ".join(indep_var_tokens[1:]) if len(indep_var_tokens) > 1 else indep_var_full
                watershed = "".join(indep_var_tokens[0]) if len(indep_var_tokens) > 1 else indep_var_full
                # Insert the metadata columns at the beginning of the DataFrame.
                # (They will be added as new columns to the row from the CSV file.)
                df.insert(0, "Region", region)
                df.insert(1, "Watershed", watershed)
                df.insert(2, "Segment Interval", segment_interval)
                df.insert(3, "Independent Variable", independent_variable)
                # Optionally, include the source file path for troubleshooting.
                df["source_file"] = filepath
                
                # Append the resulting row to the corresponding summary list.
                summary_data[file].append(df)

# Write out each summary CSV file.
for result_filename, df_list in summary_data.items():
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        # Remove 
        
        summary_filename = target_files[result_filename]
        combined_df.to_csv(summary_filename, index=False)
        print(f"Created summary file: {summary_filename} with {len(combined_df)} row(s).")
    else:
        print(f"No files found for {result_filename}.")



Processing root: CPF\Outputs\Segmented 10m\CPF sfm_deposition_logtrans, file: loocv_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_deposition_logtrans, file: tidy_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_deposition_logtrans, file: varcomp_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_erosion_logtrans, file: loocv_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_erosion_logtrans, file: tidy_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_erosion_logtrans, file: varcomp_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_net change_logtrans, file: loocv_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_net change_logtrans, file: tidy_results.csv
Processing root: CPF\Outputs\Segmented 10m\CPF sfm_net change_logtrans, file: varcomp_results.csv
Processing root: CPF\Outputs\Segmented 10m\ME sfm_deposition_logtrans, file: loocv_results.csv
Processing root: CPF\Outputs\Segmented 10m\ME s

## Aggregate Model Performance Evaluation Files

In [101]:
import os
import pandas as pd

# Define input folders and output file path
input_folders = ['CPF', 'ETF']
output_file = os.path.join('Results Summary', 'corr_0.7_evaluation_summary.csv')
variable_of_interest = 'sum'
# List to collect summary rows
summary_rows = []

for input_folder in input_folders:
    # Walk through the input folder recursively
    for root, dirs, files in os.walk(input_folder):
        # Skip any subfolder named "Archive" (case insensitive)
        dirs[:] = [d for d in dirs if d.lower() != "archive"]
        
        # Process files that end with "corr_0.7_evaluation.csv" and contain "mean" or "net" in the name
        for file in files:
            if file.endswith("corr_0.7_evaluation.csv") and (variable_of_interest in file.lower() or "net" in file.lower()) and 'norm' not in file.lower():
                print(f"Processing file: {file}")
                file_path = os.path.join(root, file)
                
                # Compute the relative path from the input folder
                rel_path = os.path.normpath(os.path.relpath(file_path, input_folder))
                rel_parts = rel_path.split(os.sep)
                
                # Ensure there are at least three folder levels: [..., segment_interval, independent_variable, file]
                if len(rel_parts) < 3:
                    continue
                
                # Extract folder names for Segment Interval and Independent Variable
                segment_interval_folder = rel_parts[-3]
                independent_variable_folder = rel_parts[-2]
                
                # Process folder names: remove first token for segment_interval and independent_variable,
                # and use first token as watershed
                seg_tokens = segment_interval_folder.split()
                segment_interval = " ".join(seg_tokens[1:]) if len(seg_tokens) > 1 else segment_interval_folder
                
                ind_tokens = independent_variable_folder.split()
                independent_variable = " ".join(ind_tokens[1:]) if len(ind_tokens) > 1 else independent_variable_folder
                watershed = ind_tokens[0] if ind_tokens else "Unknown"
                
                # The Region is taken as the base name of the input folder.
                region = os.path.basename(os.path.normpath(input_folder))
                
                try:
                    # Read the CSV file
                    df = pd.read_csv(file_path)
                    if df.empty:
                        continue
                    first_row = df.iloc[0]
                    
                    # Select the desired columns from the evaluation file
                    selected_columns = ["Bias", "RMSPE", "RAV", "AIC", 
                                        "Mean_Residuals", "SD_Residuals", 
                                        "Mean_Response", "SD_Response"]
                    selected_data = first_row[selected_columns]
                    
                    # Build a dictionary for the summary row, adding the file base name
                    row_data = {
                        "Region": region,
                        "Watershed": watershed,
                        "Segment Interval": segment_interval,
                        "Independent Variable": independent_variable,
                        "RMSPE": selected_data["RMSPE"],
                        "Mean Response": selected_data["Mean_Response"],
                        "SD Response": selected_data["SD_Response"],
                        "Mean Residuals": selected_data["Mean_Residuals"],
                        "SD Residuals": selected_data["SD_Residuals"],
                        "AIC": selected_data["AIC"],
                        "Bias": selected_data["Bias"],
                        "RAV": selected_data["RAV"],
                        "FileBaseName": os.path.splitext(file)[0]
                    }
                    summary_rows.append(row_data)
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

# Create a DataFrame from the collected rows and write to CSV
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(output_file, index=False)
    print(f"Summary saved to {output_file}")
else:
    print("No valid files found.")


Processing file: ch_sfm_deposition_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_erosion_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_net_change_corr_0.7_evaluation.csv
Processing file: ch_sfm_deposition_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_erosion_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_net_change_corr_0.7_evaluation.csv
Processing file: ch_lidar_erosion_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_deposition_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_erosion_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_net_change_corr_0.7_evaluation.csv
Processing file: ch_sfm_deposition_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_net_change_corr_0.7_evaluation.csv
Processing file: ch_lidar_erosion_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_deposition_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_erosion_sum_corr_0.7_evaluation.csv
Processing file: ch_sfm_net_change_corr_0.7_evaluation.csv
Processing file: ch_sfm_de

## Aggregate CSV files to single file

In [102]:
import pandas as pd

def join_loocv_pred_and_mod_eval(loocv_pred_csv_path, mod_eval_csv_path, output_csv_path):
    # Read the tidy data
    loocv_pred_df = pd.read_csv(loocv_pred_csv_path)
    
    # Read the varcomp data
    mod_eval_df = pd.read_csv(mod_eval_csv_path)
    
    # Make "independent variable" column in loocv_pred_df match "Independent Variable" in mod_eval_df
    # Merge on the four key fields
    merged_df = pd.merge(
        loocv_pred_df,
        mod_eval_df,
        on=["Region", "Watershed", "Segment Interval", "Independent Variable"],
        how="left"  # left-join so that all loocv_pred rows remain even if mod_eval is missing
    )
    
    # Write to a new CSV
    merged_df.to_csv(output_csv_path, index=False)

join_loocv_pred_and_mod_eval(
    loocv_pred_csv_path=r"Results Summary\loocv_prediction_summary.csv",
    mod_eval_csv_path=r"Results Summary\corr_0.7_evaluation_summary.csv",
    output_csv_path=r"Results Summary\loocv_prediction_summary.csv"
)



### loocv_prediction_summary is left out because left joining it with varcomp produces an atrocious amount of data

In [103]:
import pandas as pd

def join_multiple_ssn2_summaries(tidy_csv_path, other_summary_csv_paths, output_csv_path):
    # Read the tidy data
    merged_df = pd.read_csv(tidy_csv_path)
    # Loop through each varcomp CSV file and merge it with the current DataFrame
    for csv_path in other_summary_csv_paths:
        varcomp_df = pd.read_csv(csv_path)
        merged_df = pd.merge(
            merged_df,
            varcomp_df,
            on=["Region", "Watershed", "Segment Interval", "Independent Variable"],
            how="left"  # left-join ensures all tidy rows remain even if a join field is missing
        )
    
    # Write the final merged DataFrame to a new CSV
    merged_df.to_csv(output_csv_path, index=False)

tidy_csv_path = r"Results Summary\tidy_summary.csv"
other_summary_csv_paths = [
    r"Results Summary\corr_0.7_evaluation_summary.csv",
    r"Results Summary\loocv_summary.csv",
    r"Results Summary\varcomp_summary.csv"

]
output_csv_path = r"Results Summary\full_results_summary.csv"

join_multiple_ssn2_summaries(tidy_csv_path, other_summary_csv_paths, output_csv_path)

df = pd.read_csv(output_csv_path)

# Drop any rows where varcomp field does not match 'Covariates (PR-sq)'
df = df[df['varcomp'] == 'Covariates (PR-sq)']

# Change 'proportion' field to 'psuedo.R.squared'
df.rename(columns={'proportion': 'pseudo.R.squared'}, inplace=True)
df.rename(columns={'RMSPE_x': 'RMSPE'}, inplace=True)
df.rename(columns={'Bias_x': 'Bias'}, inplace=True)
df.rename(columns={'RAV_x': 'RAV'}, inplace=True)
df.rename(columns={'term': 'Dependent Variable'}, inplace=True)
# Drop varcomp field
df.drop(columns=['varcomp'], inplace=True)
df.drop(columns=['RMSPE_y'], inplace=True)
df.drop(columns=['Bias_y'], inplace=True)
df.drop(columns=['RAV_y'], inplace=True)

# Change any instances of 'ch_flow.accumulation.max' to 'ws_drainage_area'
df['Independent Variable'] = df['Independent Variable'].replace('ch_flow.accumulation.max', 'ws_drainage.area')
# Change any instances of 'hs_flow.accumulation.max' to 'hs_drainage_area'
df['Independent Variable'] = df['Independent Variable'].replace('hs_flow.accumulation.max', 'hs_drainage.area')
df['Independent Variable'] = df['Independent Variable'].replace('hs_ndvi.mean.mean', 'hs_ndvi.mean')
df['Independent Variable'] = df['Independent Variable'].replace('ch_valley_width', 'ch_valley.width')
df['Independent Variable'] = df['Independent Variable'].replace('ws_RV.Clay', 'ws_clay.content')
df['Independent Variable'] = df['Independent Variable'].replace('ws_RV.Sand', 'ws_sand.content')
df['Independent Variable'] = df['Independent Variable'].replace('ws_RV.Silt', 'ws_silt.content')


os.remove(output_csv_path)
df.to_csv(output_csv_path, index=False)


## Aggregate ssn results text files to single folder

In [ ]:
import os
import glob
import shutil
import argparse

def copy_files(src_dir, dst_dir):
    # Create the destination directory if it does not exist.
    os.makedirs(dst_dir, exist_ok=True)

    # Build the recursive search pattern.
    pattern = os.path.join(src_dir, '**', '*_VIF-3_corr0.7.txt')
    
    # Find all files in the source directory and its subdirectories that match the pattern.
    files_to_copy = glob.glob(pattern, recursive=True)
    print(f"Found {len(files_to_copy)} files to copy.")
    
    # Copy each file to the destination directory.
    for file_path in files_to_copy:
        if 'mean' or 'norm' in file_path:
            continue
        shutil.copy(file_path, dst_dir)
        # Add "ETF_" prefix to the copied file name.
        base_name = os.path.basename(file_path)
        parent_dir = os.path.basename(os.path.dirname(file_path))
        watershed = parent_dir.split(' ')[0]
        new_name = f"{watershed}_{base_name}"
        new_file_path = os.path.join(dst_dir, new_name)
        shutil.move(os.path.join(dst_dir, base_name), new_file_path)
        print(f"Copied: {file_path} to {dst_dir}")

src_dir = 'CPF'
dst_dir = r'Results Summary\Results txt files\CPF'
copy_files(src_dir, dst_dir)


Found 112 files to copy.
